# 01 — Basic Retrieval-Augmented Generation

**CIAL Knowledge OS** · local-only · offline-first · model-agnostic

This notebook is a learning and orchestration layer. Reusable implementation lives in `src/cial_knowledge_os`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

## 1. Objective

Build and inspect a baseline local RAG pipeline over small, non-sensitive airport documents. The experiment covers document loading, metadata-preserving chunking, local embeddings, embedded Qdrant retrieval, bounded context construction, grounded Ollama generation, citations, and timing.

Success means evidence is visible before generation, missing evidence produces a safe failure, and the notebook runs top-to-bottom from a fresh kernel.

## 2. Theory

Retrieval-Augmented Generation separates knowledge lookup from language generation:

1. Split documents into bounded, overlapping chunks.
2. Embed chunks and questions with the same local model.
3. Retrieve the nearest chunks from a local vector store.
4. Send only the most relevant evidence to the local LLM.
5. Cite that evidence or explicitly report that the answer is unavailable.

This notebook establishes a dense-retrieval baseline. Hybrid retrieval, metadata authorization, reranking, verification, and conflict handling are later improvements.

## 3. Architecture

```mermaid
flowchart LR
    A[Local TXT and PDF] --> B[Load]
    B --> C[Chunk]
    C --> D[Local embeddings]
    D --> E[Embedded Qdrant]
    Q[Question] --> F[Retrieve]
    E --> F
    F --> G[Bounded cited context]
    G --> H[Local Ollama model]
    H --> I[Grounded answer]
```

Documents, vectors, prompts, and generated text remain on the workstation. Models and paths are selected through configuration.

## 4. Implementation

### Setup

Import reusable project APIs plus matplotlib for notebook display.

In [ ]:
import matplotlib.pyplot as plt

from cial_knowledge_os import (
    BasicRAGPipeline, KnowledgeOSConfig, Timer,
    create_local_llm, create_sample_airport_documents,
    format_retrieved_context, generate_answer, load_embedding_model,
    plot_chunk_size_distribution, plot_retrieval_scores,
    plot_timing_breakdown, print_benchmark_table,
    print_retrieval_results, summarize_chunks, summarize_documents,
)

### Configuration

Use one configuration object for local paths, models, chunking, retrieval depth, and context limits.

In [ ]:
config = KnowledgeOSConfig(project_root=PROJECT_ROOT)

Display a compact configuration summary using repository-relative paths.

In [ ]:
config_summary = {
    "sample_dir": config.sample_data_dir.relative_to(PROJECT_ROOT).as_posix(),
    "pdf_dir": config.pdf_data_dir.relative_to(PROJECT_ROOT).as_posix(),
    "qdrant_dir": config.qdrant_dir.relative_to(PROJECT_ROOT).as_posix(),
    "embedding_model": config.embedding_model_name,
    "embedding_device": config.embedding_device,
    "ollama_model": config.ollama_model_name,
    "chunk_size": config.chunk_size,
    "chunk_overlap": config.chunk_overlap,
    "top_k": config.top_k,
    "max_context_chars": config.max_context_chars,
}
config_summary

### Sample Documents

Create the small versioned sample corpus when absent. Existing sample files are not overwritten.

In [ ]:
sample_paths = create_sample_airport_documents(config)
[path.relative_to(PROJECT_ROOT).as_posix() for path in sample_paths]

### Data Loading

Load sample and runtime text documents plus optional PDFs from `data/pdf/`. PDF processing is local: Docling is preferred and PyMuPDF is the fallback. With no PDFs, sample documents keep the experiment runnable.

In [ ]:
pipeline = BasicRAGPipeline(config)

Run the local loaders through the reusable pipeline.

In [ ]:
documents = pipeline.load()

Inspect document counts, character totals, loader types, and source names.

In [ ]:
summarize_documents(documents)

### Processing

Chunk documents while retaining source metadata and adding traceable chunk identifiers.

In [ ]:
chunks = pipeline.chunk()

Summarize the resulting chunk sizes and source coverage.

In [ ]:
summarize_chunks(chunks)

Inspect one chunk to confirm that content and traceability metadata were preserved.

In [ ]:
{
    "metadata": chunks[0].metadata,
    "preview": chunks[0].page_content[:240],
}

### Local Embeddings

Load the configured model strictly from the local Hugging Face cache. The helper reports the actual device and fails without downloading when the model is absent.

In [ ]:
embedding_model = load_embedding_model(config)
pipeline.embedding_model = embedding_model

Embed every chunk locally and inspect the resulting matrix shape.

In [ ]:
chunk_vectors = pipeline.embed()
chunk_vectors.shape

### Local Vector Index

Reset the experiment storage, recreate the collection, and index chunk text with metadata. Embedded Qdrant permits one process per path; restart kernels when locked or use Qdrant server mode for concurrent processes.

In [ ]:
qdrant = pipeline.index()
{
    "collection": config.qdrant_collection_name,
    "indexed_chunks": qdrant.count(config.qdrant_collection_name).count,
}

### Retrieval Experiment

Retrieve and inspect scores, source names, page numbers, chunk IDs, and previews before generation.

In [ ]:
question = "What PPE is required for electrical maintenance?"
retrieved = pipeline.retrieve(question)
print_retrieval_results(retrieved)

### Grounded Context

Build the complete bounded evidence context supplied to the local language model.

In [ ]:
context = format_retrieved_context(retrieved, config.max_context_chars)
print(context)

### Generation Experiment

Validate the configured model in the local Ollama store, then answer only from retrieved evidence. No model is downloaded automatically.

In [ ]:
try:
    local_llm = create_local_llm(config)
except RuntimeError:
    pipeline.close()
    raise

pipeline.llm = local_llm
print(f"Local Ollama model: {config.ollama_model_name}")

Generate the primary answer and record query latency. Release Qdrant immediately if generation fails.

In [ ]:
try:
    with Timer(pipeline.metrics, "generation_latency"):
        answer = generate_answer(local_llm, question, context)
except Exception:
    pipeline.close()
    raise

pipeline.metrics["total_pipeline_latency"] = (
    pipeline.metrics["retrieval_latency"]
    + pipeline.metrics["generation_latency"]
)
print(answer)

Map answer reference numbers to the exact source, page, chunk ID, and retrieval score.

In [ ]:
citation_map = [
    {
        "reference_id": rank,
        "source": item["source"],
        "page_number": item["page_number"],
        "chunk_id": item["chunk_id"],
        "score": round(item["score"], 4),
    }
    for rank, item in enumerate(retrieved, start=1)
]
citation_map

### Evaluation Questions

Run fixed questions against the same index. Evidence is printed before every answer. The final question deliberately asks for unavailable information.

In [ ]:
benchmark_metrics = pipeline.metrics.copy()

Inspect evidence and answers for the fixed evaluation set, then release the embedded Qdrant client.

In [ ]:
test_questions = [
    "How often must the runway surface be inspected?",
    "What checks are required at the start of a terminal shift?",
    "What is the approved catering budget for next year?",
]

try:
    for test_question in test_questions:
        test_results = pipeline.retrieve(test_question)
        print(f"\nQUESTION: {test_question}")
        print_retrieval_results(test_results)
        test_context = format_retrieved_context(test_results, config.max_context_chars)
        print("ANSWER:", generate_answer(local_llm, test_question, test_context))
finally:
    pipeline.close()

## 5. Visualization

Use matplotlib-only diagnostics to inspect chunk sizing, retrieval scores, and pipeline timing.

In [ ]:
plot_chunk_size_distribution(chunks)
plot_retrieval_scores(retrieved)
plot_timing_breakdown(benchmark_metrics)
plt.show()

## 6. Benchmark

The timing table reports document loading, optional PDF loading, chunking, embedding, indexing, retrieval, generation, and total query latency when available. Compare runs only on the same hardware and corpus.

Production evaluation must additionally measure correctness, retrieval relevance, citation accuracy, hallucination risk, token usage, and local hardware feasibility.

In [ ]:
print_benchmark_table(benchmark_metrics)

## 7. Advantages

- Documents, vectors, prompts, and inference remain local.
- Retrieval is independent of the configured local language model.
- Bounded context reduces token usage and unsupported generation.
- Source, page, chunk, and score metadata remain inspectable.
- Reusable code stays in importable modules instead of notebook cells.
- The same components can support later evaluation and backend services.

## 8. Limitations

- Dense retrieval can miss exact identifiers and keyword matches.
- No authorization filter, reranker, query transformation, or conflict detector is applied.
- Docling output may not retain page boundaries; PyMuPDF provides page-level text when used.
- Scanned PDFs require a separately approved local OCR configuration.
- Similarity scores are not calibrated confidence values.
- Citations are mapped but not independently verified.
- Embedded Qdrant supports one process per storage path.

## 9. Enterprise Considerations

Production ingestion must validate department, document type, asset, location, version, date, owner, access level, and retention metadata. Retrieval must enforce role and department authorization before evidence reaches generation. Real CIAL documents must remain in controlled ignored storage and must never be uploaded externally.

Operational promotion also requires version-aware indexing, duplicate detection, audit logs, encryption, backup and recovery, citation verification, weak or conflicting evidence handling, and tests against long manuals, tables, scans, acronyms, and airport terminology. Multi-process backends should use Qdrant server mode.

## 10. What We'll Improve Next

The next experiment will add local query transformations while preserving the same loaders, chunking, embeddings, vector store, retrieval schema, and grounded-generation interface. Later work will compare hybrid vector/BM25 retrieval, metadata filters, local reranking, citation verification, and contradictory-evidence handling against this baseline.